# yeast-1-4-5-8_vs_7: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('37_yeast-1-4-5-8_vs_7_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,22.079,67.246,0.000
smote,51.489,14.996,10.063,31.667,85.736,58.701,14.759,67.101,33.243
bsmote,54.059,19.544,14.122,33.333,90.265,61.799,16.124,67.298,36.801
ros,54.415,17.827,12.391,35.000,87.479,61.240,16.437,67.102,36.121
adasyn,51.167,14.783,9.921,31.667,86.124,58.895,14.945,67.380,32.975
casb,35.402,14.610,11.692,23.333,91.786,57.560,18.015,68.381,25.006
casb_relaxed,45.703,14.962,10.773,26.667,88.835,57.751,17.633,67.078,30.332
casb_edited,48.887,12.761,8.297,30.000,84.618,57.309,13.977,66.666,30.824
casb_ros,37.766,14.693,12.258,23.333,92.237,57.785,17.906,69.193,26.229


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,20.162,70.454,0.000
smote,65.312,19.055,11.682,53.333,81.442,67.388,15.887,72.135,42.183
bsmote,66.243,22.571,14.492,51.667,86.120,68.893,16.736,72.118,44.407
ros,62.964,20.080,12.847,46.667,85.519,66.093,16.752,69.661,41.522
adasyn,64.404,17.409,10.440,53.333,79.103,66.218,13.550,71.036,40.906
casb,67.912,23.356,15.010,55.000,85.822,70.411,17.937,72.567,45.634
casb_relaxed,64.700,18.421,11.226,53.333,80.839,67.086,15.251,71.554,41.561
casb_edited,65.016,17.153,10.187,56.667,77.366,67.016,11.809,70.042,41.084
casb_ros,59.455,19.513,12.879,41.667,86.804,64.235,13.388,68.880,39.484


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,19.555,71.815,0.000
smote,31.222,14.261,15.413,15.000,96.075,55.538,19.643,70.887,22.742
bsmote,12.123,6.618,14.762,5.000,98.037,51.519,16.465,69.476,9.371
ros,8.103,4.444,6.667,3.333,99.322,51.327,15.682,71.875,6.274
adasyn,25.712,12.283,14.107,11.667,96.905,54.286,16.057,69.625,18.998
casb,12.124,6.222,8.333,5.000,99.170,52.085,20.888,74.730,9.173
casb_relaxed,20.118,9.634,12.679,8.333,97.965,53.149,15.329,71.451,14.876
casb_edited,44.543,17.714,16.024,21.667,94.347,58.007,15.631,70.367,31.128
casb_ros,8.103,4.444,6.667,3.333,99.473,51.403,17.276,72.693,6.274


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,15.524,71.188,0.000
smote,62.290,15.910,9.702,55.000,73.837,64.419,15.663,71.004,39.100
bsmote,59.979,17.302,10.859,45.000,83.109,64.054,18.161,71.185,38.640
ros,60.475,15.766,9.800,53.333,73.865,63.599,14.313,69.615,38.120
adasyn,61.919,15.477,9.353,55.000,73.381,64.190,15.291,70.980,38.698
casb,59.865,17.048,10.823,46.667,81.011,63.839,19.343,71.850,38.456
casb_relaxed,64.923,18.520,11.384,55.000,80.170,67.585,16.384,72.328,41.722
casb_edited,63.300,15.876,9.439,56.667,74.522,65.594,16.229,72.290,39.588
casb_ros,58.732,17.447,11.247,43.333,83.269,63.301,18.900,71.170,38.090


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,19.330,70.176,0.000
smote,52.578,16.055,11.715,38.750,84.273,61.511,16.488,70.282,34.317
bsmote,48.101,16.509,13.559,33.750,89.383,61.566,16.871,70.019,32.305
ros,46.489,14.529,10.426,34.583,86.546,60.565,15.796,69.563,30.509
adasyn,50.801,14.988,10.955,37.917,83.878,60.897,14.961,69.755,32.894
casb,43.825,15.309,11.465,32.500,89.447,60.974,19.046,71.882,29.567
casb_relaxed,48.861,15.384,11.516,35.833,86.952,61.393,16.149,70.603,32.123
casb_edited,55.437,15.876,10.987,41.250,82.713,61.982,14.411,69.841,35.656
casb_ros,41.014,14.024,10.763,27.917,90.446,59.181,16.868,70.484,27.519


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,48.887,ros,54.415,-5.528,LOSS
1,f1,casb_relaxed,14.962,bsmote,19.544,-4.583,LOSS
2,precision,casb_ros,12.258,bsmote,14.122,-1.864,LOSS
3,recall,casb_edited,30.000,ros,35.000,-5.000,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,67.912,bsmote,66.243,1.669,WIN
1,f1,casb,23.356,bsmote,22.571,0.785,WIN
2,precision,casb,15.010,bsmote,14.492,0.519,WIN
3,recall,casb_edited,56.667,smote,53.333,3.333,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,44.543,smote,31.222,13.321,WIN
1,f1,casb_edited,17.714,smote,14.261,3.453,WIN
2,precision,casb_edited,16.024,smote,15.413,0.610,WIN
3,recall,casb_edited,21.667,smote,15.000,6.667,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,55.437,smote,52.578,2.858,WIN
1,f1,casb_edited,15.876,bsmote,16.509,-0.633,LOSS
2,precision,casb_relaxed,11.516,bsmote,13.559,-2.043,LOSS
3,recall,casb_edited,41.250,smote,38.750,2.500,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.239193,0.699248,0.000000,None
1,42,1,svm,smote,0.50,0.548408,0.190476,0.133333,0.333333,0.902256,0.617794,0.300081,0.748120,0.369442,None
2,42,1,svm,bsmote,0.50,0.674453,0.285714,0.200000,0.500000,0.909774,0.704887,0.307367,0.766917,0.480084,None
3,42,1,svm,ros,0.50,0.685511,0.352941,0.272727,0.500000,0.939850,0.719925,0.320910,0.637845,0.519226,None
4,42,1,svm,adasyn,0.75,0.660371,0.230769,0.150000,0.500000,0.872180,0.686090,0.315708,0.738095,0.445570,None
5,42,1,svm,casb,0.75,0.671660,0.272727,0.187500,0.500000,0.902256,0.701128,0.320017,0.674185,0.472194,None
6,42,1,svm,casb_relaxed,0.50,0.555221,0.222222,0.166667,0.333333,0.924812,0.629073,0.306342,0.680451,0.388722,None
7,42,1,svm,casb_edited,0.50,0.546119,0.181818,0.125000,0.333333,0.894737,0.614035,0.272269,0.699248,0.363968,None
8,42,1,svm,casb_ros,1.00,0.668856,0.260870,0.176471,0.500000,0.894737,0.697368,0.310870,0.665414,0.464863,None
9,42,1,knn,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.275826,0.763158,0.000000,None
